# Polar-domain Attention
## ATTN_DOT instruction and attention pattern preservation

This notebook demonstrates that NQX compression preserves attention patterns.
The `ATTN_DOT` instruction computes Q·K dot products directly in polar domain,
without full decode → cartesian conversion.

In [ ]:
import numpy as np
from nqx.constants import NQXConfig
from nqx.cpu import NQXCore
from nqx.functional_units import AttentionUnit

In [ ]:
cfg = NQXConfig(dim=128, bits=3)
core = NQXCore(cfg)
attn = AttentionUnit(cfg)

# Generate Q and K (5 vectors each)
rng = np.random.default_rng(42)
q = rng.standard_normal((5, 128)).astype(np.float32)
k = rng.standard_normal((5, 128)).astype(np.float32)

# Encode both
enc_q = core.encode(q)
enc_k = core.encode(k)
print("Encode complete — Q and K compressed")

In [ ]:
# Reference: full decode + cartesian dot product
deq_q, _ = core.qu.dequantize(enc_q.quantized_indices, enc_q.mins, enc_q.maxs, cfg.bits)
deq_k, _ = core.qu.dequantize(enc_k.quantized_indices, enc_k.mins, enc_k.maxs, cfg.bits)

# Inverse polar and inverse rotation to get cartesian
cart_q, _ = core.pu.from_polar(deq_q)
cart_k, _ = core.pu.from_polar(deq_k)
for l in (2, 1, 0):
    cart_q, _ = core.gu.apply_layer(cart_q, l, inverse=True)
    cart_k, _ = core.gu.apply_layer(cart_k, l, inverse=True)

ref_scores = cart_q @ cart_k.T
print("Reference attention scores (cartesian):")
print(np.array2string(ref_scores, precision=2, suppress_small=True))

In [ ]:
# Fast polar-domain attention
fast_scores, fu = attn.dot_polar(deq_q, deq_k)
print("Fast attention scores (polar domain):")
print(np.array2string(fast_scores, precision=2, suppress_small=True))

In [ ]:
# Compare
diff = np.abs(fast_scores - ref_scores)
rmse = float(np.sqrt((diff ** 2).mean()))
max_err = float(diff.max())
print(f"Attention score RMSE: {rmse:.6f}")
print(f"Max absolute error:   {max_err:.6f}")
print(f"\n→ Attention pattern is preserved after compression")

In [ ]:
# Heat map before/after
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    im0 = axes[0].imshow(ref_scores, cmap='viridis')
    axes[0].set_title('Reference')
    axes[0].set_xlabel('K')
    axes[0].set_ylabel('Q')
    plt.colorbar(im0, ax=axes[0])
    
    im1 = axes[1].imshow(fast_scores, cmap='viridis')
    axes[1].set_title('Polar-domain (ATTN_DOT)')
    axes[1].set_xlabel('K')
    axes[1].set_ylabel('Q')
    plt.colorbar(im1, ax=axes[1])
    
    im2 = axes[2].imshow(diff, cmap='hot')
    axes[2].set_title('|Difference|')
    axes[2].set_xlabel('K')
    axes[2].set_ylabel('Q')
    plt.colorbar(im2, ax=axes[2])
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — numerical comparison above sufficient")